# 06 Date Validation

Check hire dates for missing values, invalid dates, and timeline inconsistencies.

## 1. Preparation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/processed/05_invalid_values.parquet")
df_clean = df.copy()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 95,000
Columns: 35


## 2. Detection

Review hire dates and check for dates beyond the reference date used in tenure calculations.

In [2]:
reference_date = pd.Timestamp("2026-10-09")

date_report = pd.DataFrame({
    "check": [
        "Total records",
        "Missing dates",
        "Earliest hire date",
        "Latest hire date",
        "Future hire dates"
    ],
    "result": [
        len(df),
        df["hire_date"].isna().sum(),
        df["hire_date"].min().date(),
        df["hire_date"].max().date(),
        (df["hire_date"] > reference_date).sum()
    ]
})

display(date_report)

,check,result
0,Total records,95000
1,Missing dates,0
2,Earliest hire date,2010-07-15
3,Latest hire date,2026-08-08
4,Future hire dates,0


### 2.1 Tenure Consistency

Check whether employee tenure matches the hire date using the reference date of October 9, 2026.

In [3]:
reference_date = pd.Timestamp("2026-10-09")

hire_dates = df_clean["hire_date"]

calculated_tenure = (
    reference_date.year
    - hire_dates.dt.year
    - (
        (hire_dates.dt.month > reference_date.month)
        | (
            (hire_dates.dt.month == reference_date.month)
            & (hire_dates.dt.day > reference_date.day)
        )
    ).astype(int)
)

mismatch = calculated_tenure != df_clean["tenure_years"]

print(f"Matching records: {(~mismatch).sum():,}")
print(f"Mismatched records: {mismatch.sum():,}")

if mismatch.any():
    display(
        df_clean.loc[mismatch, [
            "employee_id",
            "hire_date",
            "tenure_years"
        ]].head(20)
    )

Matching records: 95,000
Mismatched records: 0


All 95,000 hire dates are valid, with no missing or future dates.

Tenure values match the hire dates for every employee, using the inferred reference date of October 9, 2026.

No corrections were needed.

## 3. Export

Save the validated dataset without modifying any values.

In [4]:
from pathlib import Path

assert mismatch.sum() == 0
assert df_clean["hire_date"].notna().all()
assert (df_clean["hire_date"] <= reference_date).all()

pd.testing.assert_frame_equal(df, df_clean)

output_path = Path("data/processed/06_date_validation.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(output_path, index=False)

saved = pd.read_parquet(output_path)
pd.testing.assert_frame_equal(df_clean, saved)

print(f"Saved: {output_path}")
print(f"Rows: {len(saved):,}")
print(f"Columns: {len(saved.columns)}")

Saved: data/processed/06_date_validation.parquet
Rows: 95,000
Columns: 35
